# Lab 02: Full TinyStories Cache + Batch Sweep on L40S

This notebook is the next practical step after the 1M-token smoke run.

Goal:

1. Build the **full TinyStories token cache** instead of the tiny smoke cache.
2. Run a **batch-size sweep** while keeping effective batch size constant.
3. Choose the fastest stable setting before the longer AdamW vs Muon benchmark.

Vocabulary:

- **Token cache**: tokenized text saved as `.bin` files so training does not re-tokenize text every run.
- **Micro-batch size**: how many sequences the GPU processes in one forward/backward pass.
- **Gradient accumulation**: doing several micro-batches before one optimizer update.
- **Effective batch tokens**: total tokens used for one optimizer update.

For this sweep we keep this fixed:

```text
effective_batch_tokens = 65,536
```

That way we are testing GPU efficiency without changing the optimizer's basic training behavior.


In [ ]:
# These paths are for the EC2 machine, not your local Mac.
REMOTE_PROJECT = '/home/anubrat/lean-ml-visual-labs'
FULL_CACHE = '/home/anubrat/pretrain_data/tinystories_gpt2_512_full'
SWEEP_ROOT = '/home/anubrat/pretrain_runs/batch_sweep_60m_adamw_fullts_001'

print('Project:', REMOTE_PROJECT)
print('Full TinyStories cache:', FULL_CACHE)
print('Batch sweep output:', SWEEP_ROOT)


## 1. Confirm GPU and environment

Run this on the EC2 Jupyter kernel. It confirms that PyTorch sees the L40S.


In [ ]:
import torch, os, subprocess, json, pathlib
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
    print('memory GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print('cwd:', os.getcwd())


## 2. Build the full TinyStories token cache

This creates:

```text
train.bin
validation.bin
metadata.json
```

The patched code streams tokens to disk instead of keeping the whole dataset in memory.

If this cell is interrupted, rerun it. It rebuilds the cache safely.


In [ ]:
cmd = f'''
cd {REMOTE_PROJECT} && \
python -u gpu_benchmark/train_gpt.py \
  --run-id prepare_full_tinystories \
  --run-dir /tmp/prepare_full_tinystories \
  --dataset tinystories \
  --data-dir {FULL_CACHE} \
  --tokenizer openai-community/gpt2 \
  --model-config minigpt_dense_60m_v1 \
  --prepare-data \
  --prepare-only
'''
print(cmd)
# Uncomment to run from inside Jupyter:
# subprocess.run(cmd, shell=True, check=True)


## 3. Inspect the full cache

This tells us exactly how many train/validation tokens we have.


In [ ]:
from pathlib import Path
meta_path = Path(FULL_CACHE) / 'metadata.json'
if meta_path.exists():
    meta = json.loads(meta_path.read_text())
    print(json.dumps(meta, indent=2))
    print('train.bin GB:', round((Path(FULL_CACHE) / 'train.bin').stat().st_size / 1024**3, 3))
    print('validation.bin GB:', round((Path(FULL_CACHE) / 'validation.bin').stat().st_size / 1024**3, 3))
else:
    print('Cache not built yet:', meta_path)


## 4. Run the batch sweep

We test micro-batches:

```text
4, 8, 16, 32, 64, 128
```

For each one, the script adjusts `grad_accum_steps` so the effective batch remains 65,536 tokens/update.

Example:

```text
micro_batch_size = 4
grad_accum_steps = 32
4 * 512 * 32 = 65,536 tokens/update

micro_batch_size = 64
grad_accum_steps = 2
64 * 512 * 2 = 65,536 tokens/update
```


In [ ]:
cmd = f'''
cd {REMOTE_PROJECT} && \
python -u gpu_benchmark/batch_sweep.py \
  --data-dir {FULL_CACHE} \
  --run-root {SWEEP_ROOT} \
  --optimizer adamw \
  --micro-batches 4,8,16,32,64,128 \
  --effective-batch-tokens 65536 \
  --steps 30 \
  --validation-tokens 16384
'''
print(cmd)
# Uncomment to run from inside Jupyter:
# subprocess.run(cmd, shell=True, check=True)


## 5. Read the sweep summary

The main number to inspect is `median_tokens_per_second`.

Also inspect `max_gpu_allocated_gb` to see how much of the L40S memory we used.


In [ ]:
import pandas as pd
summary_path = Path(SWEEP_ROOT) / 'batch_sweep_summary.csv'
if summary_path.exists():
    df = pd.read_csv(summary_path)
    display(df)
    display(df.sort_values('median_tokens_per_second', ascending=False).head(3))
else:
    print('Sweep summary not found yet:', summary_path)


## 6. Decision rule

Choose the largest/fastest micro-batch that:

1. does not OOM,
2. has high tokens/sec,
3. leaves some VRAM safety margin,
4. keeps effective batch tokens at 65,536 for the AdamW vs Muon benchmark.

After this, we run:

```text
AdamW 30-60 min on full TinyStories
Hybrid Muon 30-60 min on full TinyStories
```

Then we compare validation loss vs tokens and validation loss vs FLOPs.
